# 🌤️ EDA — Weather ETL Pipeline

**Mục tiêu**: Hiểu cấu trúc data từ Open-Meteo API trước khi viết Transform.

**Data**: 63 tỉnh thành Việt Nam × 24 giờ/ngày = 1,512 records/lần chạy

---

In [12]:
import sys
import os

# Thêm project root vào path để import được src.*
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
from src.extract import extract_all_locations

print('Setup xong!')

Setup xong!


## 1. Lấy data từ API

In [13]:
df = extract_all_locations()
print(f'Lay xong: {len(df)} records')

2026-08-25 22:53:29 | INFO     | src.extract | === START EXTRACT | 63 tinh thanh | past_days=1, forecast_days=0 → 24 records/tinh ===
2026-08-25 22:53:29 | INFO     | src.extract | Fetching: Ha Noi
2026-08-25 22:53:30 | INFO     | src.extract | [Ha Noi] OK - 24 records
2026-08-25 22:53:30 | INFO     | src.extract | Fetching: Hai Phong
2026-08-25 22:53:31 | INFO     | src.extract | [Hai Phong] OK - 24 records
2026-08-25 22:53:31 | INFO     | src.extract | Fetching: Quang Ninh
2026-08-25 22:53:32 | INFO     | src.extract | [Quang Ninh] OK - 24 records
2026-08-25 22:53:32 | INFO     | src.extract | Fetching: Bac Giang
2026-08-25 22:53:33 | INFO     | src.extract | [Bac Giang] OK - 24 records
2026-08-25 22:53:33 | INFO     | src.extract | Fetching: Bac Kan
2026-08-25 22:53:34 | INFO     | src.extract | [Bac Kan] OK - 24 records
2026-08-25 22:53:34 | INFO     | src.extract | Fetching: Bac Ninh
2026-08-25 22:53:35 | INFO     | src.extract | [Bac Ninh] OK - 24 records
2026-08-25 22:53:35 | IN

## 2. Cấu trúc cơ bản

In [14]:
print(f'Shape  : {df.shape}  →  {df.shape[0]} rows × {df.shape[1]} cols')
print(f'Columns: {df.columns.tolist()}')
print()
print('Data types:')
print(df.dtypes)

Shape  : (1512, 5)  →  1512 rows × 5 cols
Columns: ['location_name', 'temperature', 'humidity', 'wind_speed', 'recorded_at']

Data types:
location_name               str
temperature             float64
humidity                  int64
wind_speed              float64
recorded_at      datetime64[us]
dtype: object


## 3. Xem 5 dòng đầu

In [15]:
df.head(10)

,location_name,temperature,humidity,wind_speed,recorded_at
0,Ha Noi,25.1,96,8.0,2026-08-24 00:00:00
1,Ha Noi,25.8,95,10.5,2026-08-24 01:00:00
2,Ha Noi,25.6,95,13.9,2026-08-24 02:00:00
3,Ha Noi,25.4,94,17.7,2026-08-24 03:00:00
4,Ha Noi,25.4,96,20.4,2026-08-24 04:00:00
5,Ha Noi,25.3,96,21.1,2026-08-24 05:00:00
6,Ha Noi,25.1,98,18.6,2026-08-24 06:00:00
7,Ha Noi,25.9,96,20.0,2026-08-24 07:00:00
8,Ha Noi,26.6,93,20.2,2026-08-24 08:00:00
9,Ha Noi,27.3,92,18.0,2026-08-24 09:00:00


## 4. Thống kê mô tả (describe)

In [16]:
df.describe()

,temperature,humidity,wind_speed,recorded_at
count,1512.000000,1512.000000,1512.000000,1512
mean,27.954101,81.678571,14.699802,2026-08-24 11:30:00
min,17.900000,34.000000,0.000000,2026-08-24 00:00:00
25%,26.000000,72.000000,9.375000,2026-08-24 05:45:00
50%,27.500000,87.000000,14.700000,2026-08-24 11:30:00
75%,29.500000,93.000000,20.100000,2026-08-24 17:15:00
max,38.700000,100.000000,34.900000,2026-08-24 23:00:00
std,3.146838,15.051154,7.237561,NaN


## 5. Missing values

In [17]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

result = pd.DataFrame({
    'missing_count': missing,
    'missing_pct_%': missing_pct
})

print('Missing values:')
result

Missing values:


,missing_count,missing_pct_%
location_name,0,0.0
temperature,0,0.0
humidity,0,0.0
wind_speed,0,0.0
recorded_at,0,0.0


## 6. Duplicate records

In [18]:
dup_total    = df.duplicated().sum()
dup_loc_time = df.duplicated(subset=['location_name', 'recorded_at']).sum()

print(f'Duplicate rows (toan bo)           : {dup_total}')
print(f'Duplicate (location + recorded_at) : {dup_loc_time}')

Duplicate rows (toan bo)           : 0
Duplicate (location + recorded_at) : 0


## 7. Số records theo tỉnh

In [19]:
counts = df.groupby('location_name').size().reset_index(name='record_count')

print(f'Tat ca tinh deu co 24 records: {(counts["record_count"] == 24).all()}')
print(f'Min: {counts["record_count"].min()}  |  Max: {counts["record_count"].max()}')
print()

# Tỉnh không đủ 24 records
not_24 = counts[counts['record_count'] != 24]
if len(not_24) > 0:
    print('CANH BAO - Tinh khong du 24 records:')
    display(not_24)
else:
    print('Tat ca 63 tinh deu du 24 records!')

counts

Tat ca tinh deu co 24 records: True
Min: 24  |  Max: 24

Tat ca 63 tinh deu du 24 records!


,location_name,record_count
0,An Giang,24
1,Ba Ria Vung Tau,24
2,Bac Giang,24
3,Bac Kan,24
4,Bac Lieu,24
...,...,...
58,Tra Vinh,24
59,Tuyen Quang,24
60,Vinh Long,24
61,Vinh Phuc,24


## 8. Khoảng giá trị & Outlier

In [20]:
print('=== Temperature (°C) ===')
print(f'  Min : {df.temperature.min()}  |  Max : {df.temperature.max()}  |  Mean : {df.temperature.mean():.1f}')
print(f'  Bất thường < 10°C : {(df.temperature < 10).sum()} records')
print(f'  Bất thường > 45°C : {(df.temperature > 45).sum()} records')

print()
print('=== Humidity (%) ===')
print(f'  Min : {df.humidity.min()}  |  Max : {df.humidity.max()}  |  Mean : {df.humidity.mean():.1f}')
print(f'  Bất thường < 0   : {(df.humidity < 0).sum()} records')
print(f'  Bất thường > 100 : {(df.humidity > 100).sum()} records')

print()
print('=== Wind speed (km/h) ===')
print(f'  Min : {df.wind_speed.min()}  |  Max : {df.wind_speed.max()}  |  Mean : {df.wind_speed.mean():.1f}')
print(f'  Bất thường < 0   : {(df.wind_speed < 0).sum()} records')
print(f'  Bất thường > 100 : {(df.wind_speed > 100).sum()} records')

=== Temperature (°C) ===
  Min : 17.9  |  Max : 38.7  |  Mean : 28.0
  Bất thường < 10°C : 0 records
  Bất thường > 45°C : 0 records

=== Humidity (%) ===
  Min : 34  |  Max : 100  |  Mean : 81.7
  Bất thường < 0   : 0 records
  Bất thường > 100 : 0 records

=== Wind speed (km/h) ===
  Min : 0.0  |  Max : 34.9  |  Mean : 14.7
  Bất thường < 0   : 0 records
  Bất thường > 100 : 0 records


## 9. Timezone check (recorded_at)

In [21]:
print(f'Min time    : {df.recorded_at.min()}')
print(f'Max time    : {df.recorded_at.max()}')
print(f'Timezone    : {df.recorded_at.dt.tz}  ← None = naive datetime, chua co tz info')
print(f'Unique days : {df.recorded_at.dt.date.nunique()} ngay')
print(f'Unique hours: {sorted(df.recorded_at.dt.hour.unique().tolist())}')

Min time    : 2026-08-24 00:00:00
Max time    : 2026-08-24 23:00:00
Timezone    : None  ← None = naive datetime, chua co tz info
Unique days : 1 ngay
Unique hours: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]


## 10. Top 5 nóng nhất & lạnh nhất hôm nay

In [22]:
# Nhiệt độ trung bình theo tỉnh
avg_temp = df.groupby('location_name')['temperature'].mean().round(1).reset_index()
avg_temp.columns = ['tinh', 'nhiet_do_tb']

print('TOP 5 NÓNG NHẤT:')
display(avg_temp.nlargest(5, 'nhiet_do_tb').reset_index(drop=True))

print()
print('TOP 5 MÁT NHẤT:')
display(avg_temp.nsmallest(5, 'nhiet_do_tb').reset_index(drop=True))

TOP 5 NÓNG NHẤT:


,tinh,nhiet_do_tb
0,Da Nang,33.6
1,Quang Ngai,33.1
2,Khanh Hoa,32.2
3,Ninh Thuan,32.2
4,Quang Nam,32.0



TOP 5 MÁT NHẤT:


,tinh,nhiet_do_tb
0,Lam Dong,20.0
1,Gia Lai,22.2
2,Dak Nong,23.5
3,Kon Tum,24.7
4,Lai Chau,25.9


---
## 📋 Kết luận EDA

| Kiểm tra | Kết quả | Cần xử lý |
|---|---|---|
| Missing values | 0 missing | Không |
| Duplicate records |0 duplicate | Không |
| Records per tỉnh | 24/24 đồng đều | Không |
| Outlier temperature |  17–39°C hợp lý | Không |
| Outlier humidity | 34–100% hợp lý | Không |
| Outlier wind_speed | 0–35 km/h hợp lý | Không |
| Timezone |Naive datetime | **Cần gán UTC+7 ở Transform** |

### → Transform sẽ tập trung vào:
1. Gán timezone `Asia/Ho_Chi_Minh` cho `recorded_at`
2. Đảm bảo đúng kiểu dữ liệu (humidity → int, round floats)
3. Map `location_name` → `location_id` (để insert vào `weather_fact`)